## License
Copyright 2026 jphall@gwu.edu. MIT License; see the repository LICENSE file.

# Text to SQL with a local credit dataset

This notebook uses few-shot examples to translate a plain-English question into a simple SQLite SELECT query, then executes it locally. 

The data is `data/credit_line_increase.csv`. 


## 1. Import packages, load data, and configure Azure


In [4]:
# 1. Import packages, load data, and configure Azure
# Create an in-memory SQLite table from the CSV before querying it.

from pathlib import Path
import os, sqlite3
from getpass import getpass
import pandas as pd
from openai import AzureOpenAI

ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists(): ROOT = ROOT.parent
data = pd.read_csv(ROOT / "data" / "credit_line_increase.csv")
connection = sqlite3.connect(":memory:")
data.to_sql("credit_accounts", connection, index=False, if_exists="replace")

RESOURCE = "gw-sb-01"
ENDPOINT = f"https://{RESOURCE}.openai.azure.com/"

api_key = os.getenv("GW_AZURE_OPENAI_KEY") or getpass("Azure OpenAI API key: ")
client = AzureOpenAI(azure_endpoint=ENDPOINT, api_key=api_key, api_version="2025-03-01-preview")

display(data.head())


,ID,LIMIT_BAL,SEX,RACE,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DELINQ_NEXT
0,1,20000,2,1.0,2,1,24,2,2,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2.0,2,2,26,-1,2,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,3.0,2,2,34,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,4.0,2,1,37,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,3.0,2,1,57,-1,0,-1,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


## 2. Generate a read-only SQL query


In [5]:
# 2. Generate a read-only SQL query
# Use short few-shot mappings to show how questions become SQL.

examples = """Question: How many accounts are in the dataset?
SQL: SELECT COUNT(*) AS account_count FROM credit_accounts;

Question: What is the average credit limit?
SQL: SELECT AVG(LIMIT_BAL) AS average_credit_limit FROM credit_accounts;
"""

question = "What is the average credit limit for each education category?"

prompt = f"""Write one SQLite SELECT statement for credit_accounts. Use only these columns: {', '.join(data.columns)}. Do not modify data. Return SQL only.

{examples}

Question: {question}

SQL:
"""

# GPT-5 uses tokens for internal reasoning; leave room for the visible SQL statement.
response = client.responses.create(
    model="gpt-5-mini",
    input=prompt,
    max_output_tokens=1200,
    reasoning={"effort": "minimal"},
)

# Remove Markdown fences if the model adds them despite the SQL-only instruction.
sql = response.output_text.strip().removeprefix("```sql").removeprefix("```").removesuffix("```").strip()

# Stop before execution rather than trying to run an empty query.
if not sql:
    raise RuntimeError("The LLM returned no visible SQL. Please run this question again.")

print(sql)


SELECT EDUCATION, AVG(LIMIT_BAL) AS average_credit_limit
FROM credit_accounts
GROUP BY EDUCATION;


## 3. Validate and run the SQL query


In [6]:
# 3. Validate and run the SQL query
# Reject non-SELECT statements before sending SQL to SQLite.

if not sql.upper().startswith("SELECT") or ";" in sql.rstrip(";"):
    raise ValueError("Only one read-only SELECT statement is allowed.")
pd.read_sql_query(sql, connection)


,EDUCATION,average_credit_limit
0,0,217142.857143
1,1,212956.069910
2,2,147062.437634
3,3,126550.270490
4,4,220894.308943
5,5,168164.285714
6,6,148235.294118
